<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/Weekly_Entry_etf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta

In [50]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
#import pandas_ta as ta
import numpy as np
import time
import ta
print("Libraries Installed!")

0.2.56
Libraries Installed!


In [51]:
# List of ETFs to analyze
df_o = pd.read_csv('etf_list.csv')
etfs = df_o['ETF'].to_list()
#etfs =['FEZ', 'VGK', 'EUFN','EWH','ICOP', 'EZA']
print(etfs)

print(len(etfs))

['SMOG', 'FEZ', 'VGK', 'SPDW', 'PIO', 'VEA', 'EWS', 'EFA', 'IEV', 'DGRE', 'VEU', 'ACWX', 'EWJ', 'EWQ', 'VXUS', 'IXUS', 'CWI', 'EPU', 'IAU', 'IAUM', 'GLD', 'GLDM', 'OUNZ', 'RING', 'GDX', 'GDXJ', 'IBND', 'IGOV', 'BWX', 'PICB', 'ISHG', 'BWZ', 'KXI', 'VDC', 'IYK']
35


In [52]:
# inspect dataframe
df_o.head()

,ETF,score
0,SMOG,0.40
1,FEZ,0.39
2,VGK,0.38
3,SPDW,0.37
4,PIO,0.37


In [53]:
# Function to fetch historical weekly data

def get_monthly_data(ticker):
  try:
      df = yf.download(ticker, period="2y", interval="1mo")
      df['10_month_SMA'] = df['Close'].rolling(window=10).mean()
      #df['above_10_month_SMA'] = df['Close'] > df['10_month_SMA']  # Convert to boolean explicitly
      # Calculate monthly ROC (based on 3 trading months)
      df['ROC_1M'] = (df['Close'].pct_change(periods=4)) *100
      # Check if ROC is positive
      df["ROC_1M_Positive"] = df["ROC_1M"] > 0
      df['ROC_1M_SMA'] = df['ROC_1M'].rolling(window=3).mean()
      # We take the difference between ROC of the current month and the ROC of the past N months
      df["ROC_1M_Slope_Positive"] = np.where(df['ROC_1M_SMA'] > df['ROC_1M_SMA'].shift(1), True, False)
      #df["ROC_1M"].rolling(window=8).apply(
        #lambda x: (x[-1] > x[0]), raw=True).astype(bool)  # Convert to boolean explicitly
      return df
  except Exception as e:
      print("There is an error getting monthly data", e)

def get_weekly_data(ticker):
  try:
      df = yf.download(ticker, period="2y", interval="1wk")
      df['20_week_SMA'] = df['Close'].rolling(window=20).mean()
      df['50_week_SMA'] = df['Close'].rolling(window=50).mean()
      df['RSI'] = compute_rsi(df['Close'])
      df['ATR'] = compute_atr(df, 14)
      df['OBV'] = compute_obv(df)
      df['10_week_avg_volume'] = df['Volume'].rolling(window=10).mean()
      df['ma'] = calculate_ma(df['RSI'])
      df['upper_band'], df['lower_band'] = calculate_bollinger_bands(df['RSI'])
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      # Compute raw EFI
      df['EFI'] = (df['Close'].diff()) * df['Volume']
      # Compute EMA of EFI
      df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
      # Determine if EFI_EMA is rising or falling
      df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')

      # Calculate weekly ROC (based on 9 trading  week)
      df['ROC_1W'] = (df['Close'].pct_change(periods=9)) *100
      # Check if ROC is positive
      df["ROC_1W_Positive"] = df["ROC_1W"] > 0
      df['ROC_1W_SMA'] = df['ROC_1W'].rolling(window=3).mean()
      # We take the difference between ROC of the current week and the ROC of the past N weeks
      df["ROC_1W_Slope_Positive"] = np.where(df['ROC_1W_SMA'] > df['ROC_1W_SMA'].shift(1), True, False)
      return df
  except Exception as e:
      print("There is an error getting weekly data", e)

def is_macd_bullish(macd_line: pd.Series, signal_line: pd.Series) -> bool:
    """
    Determines if there is a bullish signal on the MACD indicator.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - macd_line (pd.Series): The MACD line values.
    - signal_line (pd.Series): The signal line values.

    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    if len(macd_line) < 3 or len(signal_line) < 3:
        return False  # Not enough data to evaluate

    # Check if the MACD line is above the signal line
    if macd_line.iloc[-1] > signal_line.iloc[-1]:
      return True
    elif macd_line.iloc[-2] <= signal_line.iloc[-2]:
      # Check if the difference between MACD and signal line is increasing
      diff_now = macd_line.iloc[-1] - signal_line.iloc[-1]
      diff_prev = macd_line.iloc[-2] - signal_line.iloc[-2]
      diff_earlier = macd_line.iloc[-3] - signal_line.iloc[-3]
      return (diff_now > diff_earlier) or (diff_now > diff_prev)
    else:
        return False



def calculate_vwap(df):
  """ Calculates vwap"""
  try:
    Typical_Price = (df['Close'].values + df['High'].values + df['Low'].values) / 3
    TPV = Typical_Price * df['Volume'].values
    vwap = TPV.cumsum() / df['Volume'].values.cumsum()
    return vwap
  except Exception as e:
    print("Something went wrong while computing the VWAP:", e)
    return None

# Function to compute RSI
def compute_rsi(series, period=10):
  try:
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))
  except Exception as e:
        print("Something went wrong while computing the RSI:", e)
        return None

def calculate_ma(data, length=10, ma_type="WMA"):
    if ma_type == "SMA":
        return data.rolling(window=length).mean()
    elif ma_type == "EMA":
        return data.ewm(span=length, adjust=False).mean()
    elif ma_type == "WMA":
        weights = np.arange(1, length+1)
        return data.rolling(length).apply(lambda x: np.dot(x, weights)/weights.sum(), raw=True)
    elif ma_type == "VWMA":
        return ta.volume_weighted_average_price(data, length)

def calculate_bollinger_bands(data, length=10, std_dev=2.0):
    sma = data.rolling(window=length).mean()
    std_dev = data.rolling(window=length).std()
    upper_band = sma + (std_dev * std_dev)
    lower_band = sma - (std_dev * std_dev)
    return upper_band, lower_band

# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)


# Function to compute On-Balance Volume (OBV)
def compute_obv(df):
  try:
    # Calculate daily price change: 1 if price is up, -1 if down, 0 if unchanged
    price_change = df['Close'].diff()

    # Use price change to decide whether to add or subtract volume
    obv = (price_change > 0).astype(int) * df['Volume']  # Volume when price goes up
    obv -= (price_change < 0).astype(int) * df['Volume']  # Volume when price goes down

    # We accumulate the OBV by taking the cumulative sum of the volume changes
    obv = obv.cumsum()

    return obv
  except Exception as e:
      print("Something went wrong while computing the OBV", e)


# Function to calculate risk-reward ratio
def calculate_risk_reward(df):
  try:
    if df.empty or len(df) < 20:  # Ensure there are enough data points
        return np.nan

    latest_price = df['Close'].iloc[-1].iloc[0]

    # Use the ATR for setting support level
    atr = df['ATR'].iloc[-1]  # Latest ATR value
    atr_multiple = 1.25  # You can adjust this multiplier based on your strategy

    # Calculate the support level using the ATR
    trailing = atr * atr_multiple
    support_level = latest_price - trailing
    resistance_level = latest_price + (2*trailing)
    risk = latest_price- support_level
    reward = resistance_level - latest_price

    # Ensure risk is greater than zero before division
    if risk > 0:
        risk_reward_ratio = reward / risk
        return risk_reward_ratio if risk_reward_ratio > 0 else np.nan , support_level, resistance_level, latest_price, trailing
    else:
        return np.nan,np.nan, np.nan, np.nan, np.nan
  except Exception as e:
      print("Something went wrong while computing the reward-risk ratio", e)

# Function to fetch daily data
def get_daily_data(ticker):
    df = yf.download(ticker, period="1y", interval="1d")
    df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
    df['20_day_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
    df['13_day_EMA'] = df['Close'].ewm(span=13, adjust=False).mean()
    df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
    df["Distance_EMA"] = (df['Close']/ df['Close'].ewm(span=20, adjust=False).mean() ) - 1
    df['RSI'] = compute_rsi(df['Close'],period=10)
    df['ATR'] = compute_atr(df, 20)
    df['ma'] = calculate_ma(df['RSI'])
    df['upper_band'], df['lower_band'] = calculate_bollinger_bands(df['RSI'])
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['VWAP'] = calculate_vwap(df)
    # Compute raw EFI
    df['EFI'] = (df['Close'].diff()) * df['Volume']
    # Compute EMA of EFI
    df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
    # Determine if EFI_EMA is rising or falling
    df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
    # Calculate daily ROC (based on 10 trading days per week)
    df['ROC'] = (df['Close'].pct_change(periods=10)) *100
    # Check if ROC is positive
    df["ROC_Positive"] = df["ROC"] > 0
    df['ROC_SMA'] = df['ROC'].rolling(window=3).mean()
    # We take the difference between ROC of the current day and the ROC of the past N days
    df["ROC_Slope_Positive"] = np.where(df['ROC_SMA'] > df['ROC_SMA'].shift(1), True, False)
    return df


# Function to check monthly trend
def is_monthly_trend_bullish(df):
    if df.empty:
        return False

    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['10_month_SMA'].iloc[-1]
    roc_above_zero = df['ROC_1M_Positive'].iloc[-1]
    roc_trend_ok = df['ROC_1M_Slope_Positive'].iloc[-1]
    above_10_month_SMA = latest_price > latest_sma
    return above_10_month_SMA and roc_above_zero and roc_trend_ok

# Function to check weekly trend
def is_weekly_trend_bullish(df):
    if df.empty:
        return False

    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['20_week_SMA'].iloc[-1]
    above_20_week_SMA = latest_price > latest_sma
    #latest_rsi = df['RSI'].iloc[-1]
    macd_bullish_signal = is_macd_bullish(df['MACD_Line'], df['Signal_Line'])
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Rising'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] > 0
    rocw_above_zero = df['ROC_1W_Positive'].iloc[-1]
    rocw_trend_ok = df['ROC_1W_Slope_Positive'].iloc[-1]
    volume_ok = df['Volume'].iloc[-1] > df['10_week_avg_volume'].iloc[-1] # Institutional interest
    volume_ok = volume_ok.iloc[0]
    # Calculate the OBV Moving Average
    df['OBV_EMA'] = df['OBV'].ewm(span=10, adjust=False).mean()
    # OBV trending up if current OBV is above the 20-period EMA
    obv_trending_up = df['OBV'].iloc[-1] > df['OBV_EMA'].iloc[-1]
    # obv_trending_up = df['OBV'].iloc[-1] > df['OBV'].iloc[-5] # OBV increasing over last 5 weeks

    # OBV trending down if current OBV is below the 20-period EMA
    obv_trending_down = df['OBV'].iloc[-1] < df['OBV_EMA'].iloc[-1]
    trend_ok =above_20_week_SMA
    rsi_ok = df['RSI'].iloc[-1] >= 50 # Not  oversold
    time.sleep(2)  # Add a delay of 1 second between requests



    return  trend_ok and rsi_ok and rocw_above_zero and (volume_ok or obv_trending_up) \
            and elderforce_trend_ok and elderforce_ema_ok

# Function to check daily entry signal
def is_daily_entry_signal(df):
    if df.empty:
        return False

    latest_price = df['Close'].iloc[-1].iloc[0]
    prev_price = df['Close'].iloc[-2].iloc[0]
    latest_sma = df['20_day_SMA'].iloc[-1]
    latest_50sma = df['50_day_SMA'].iloc[-1]
    above_50SMA = latest_price > latest_50sma
    latest_rsi = df['RSI'].iloc[-1]
    latest_distance_20ema = df['Distance_EMA'].iloc[-1]
    macd_bullish_signal = is_macd_bullish(df['MACD_Line'], df['Signal_Line'])
    vwap_price = df['VWAP'].iloc[-1]
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Rising'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] > 0
    rocd_above_zero = df['ROC_Positive'].iloc[-1]
    rocd_trend_ok = df['ROC_Slope_Positive'].iloc[-1]

    # Look for a breakout above 20-day SMA & RSI > 50
    return above_50SMA and (latest_rsi > 50) \
            and elderforce_trend_ok or elderforce_ema_ok or (latest_price > vwap_price)

# Check entry conditions
def check_entry_conditions(tickers):
    results = []
    for ticker in tickers:
      df = get_daily_data(ticker)
      latest_price = df['Close'].iloc[-1].iloc[0]
      prev_price = df['Close'].iloc[-2].iloc[0]
      latest_sma = df['50_day_SMA'].iloc[-1]
      latest_rsi = df['RSI'].iloc[-1]
      latest_distance_20ema = df['Distance_EMA'].iloc[-1]
      latest_price_8ema =df['8_day_EMA'].iloc[-1]
      latest_price_13ema =df['13_day_EMA'].iloc[-1]
      latest_price_21ema =df['21_day_EMA'].iloc[-1]


      if latest_price >= latest_price_8ema:
        entry_signal = "Momentum Entry"
      elif (latest_price < latest_price_8ema) and (latest_price >= latest_price_13ema):
        entry_signal = "Pullback Entry"
      elif (latest_price <= latest_price_13ema) and (latest_price >= latest_price_21ema):
          entry_signal= "Below Pullback Entry"
      elif  (latest_price >= latest_sma):
          entry_signal = "Weak but still Bullish"
      else:
        entry_signal = "Bearish"
      results.append([ticker, entry_signal])
    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["ETF", "Entry_Signal"])
    return df_results

# Multi-timeframe strategy check returning a DataFrame
def check_mtf_entry(tickers):
    results = []

    for ticker in tickers:
        monthly_df = get_monthly_data(ticker)
        weekly_df = get_weekly_data(ticker)
        daily_df = get_daily_data(ticker)

        if is_monthly_trend_bullish(monthly_df) and is_weekly_trend_bullish(weekly_df):
        #if  is_weekly_trend_bullish(weekly_df):
            if is_daily_entry_signal(daily_df):
                entry_signal = "Entry Confirmed ✅"
            else:
                entry_signal = "No Entry Yet on Daily Timeframe ⏳"
        else:
            entry_signal = "Monthly or Weekly Trend Not Bullish ❌"

        results.append([ticker, entry_signal])
        time.sleep(2)  # Add a delay of 1 second between requests

    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["ETF", "Entry_Signal"])
    return df_results



In [54]:
# Apply TA filters and prioritize ETFs
#results = []
#for etf in etfs:
  #df =get_weekly_data(etf)
  #price = df['Close'].iloc[-1].iloc[0]
  #above_20SMA = price > df['20_week_SMA'].iloc[-1]
  #above_50SMA = price > df['50_week_SMA'].iloc[-1]
  #rsi_ok = df['RSI'].iloc[-1] >= 50 # Not  oversold
  #volume_ok = df['Volume'].iloc[-1] > df['10_week_avg_volume'].iloc[-1] # Institutional interest
  #volume_ok = volume_ok.iloc[0]
  #print(volume_ok)

  # Calculate the OBV Moving Average
  #df['OBV_EMA'] = df['OBV'].ewm(span=10, adjust=False).mean()

  # OBV trending up if current OBV is above the 20-period EMA
  #obv_trending_up = df['OBV'].iloc[-1] > df['OBV_EMA'].iloc[-1]
  # obv_trending_up = df['OBV'].iloc[-1] > df['OBV'].iloc[-5] # OBV increasing over last 5 weeks

  # OBV trending down if current OBV is below the 20-period EMA
  #obv_trending_down = df['OBV'].iloc[-1] < df['OBV_EMA'].iloc[-1]

  #trend_ok = above_20SMA


  #if trend_ok and rsi_ok and (volume_ok or obv_trending_up):
    #print(f" {etf} passes the first check on weekly timeframe!")
    #results.append({"ETF": etf })
  #else:
    #print(f" {etf} does not pass the first check on weekly timeframe!")
  #time.sleep(2)  # Add a delay of 1 second between requests


# Multi-time frame entry Check
#df_results = pd.DataFrame(results).dropna()
etfs_to_check = df_o['ETF'] #df_results['ETF'].tolist()
df_signals = check_mtf_entry(etfs_to_check)

df_signals.head()



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Entry_Signal
0,SMOG,Entry Confirmed ✅
1,FEZ,Entry Confirmed ✅
2,VGK,Entry Confirmed ✅
3,SPDW,Entry Confirmed ✅
4,PIO,Monthly or Weekly Trend Not Bullish ❌


## Generate buy list

In [62]:
df_final = df_signals[df_signals['Entry_Signal'] =="Entry Confirmed ✅"]
final_etfs_to_check = df_final['ETF'].tolist()


buy_list = check_entry_conditions(final_etfs_to_check)

#buy_list.head()
buy_list

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Entry_Signal
0,SMOG,Momentum Entry
1,FEZ,Momentum Entry
2,VGK,Momentum Entry
3,SPDW,Momentum Entry
4,VEA,Momentum Entry
5,EWS,Momentum Entry
6,EFA,Momentum Entry
7,IEV,Momentum Entry
8,VEU,Momentum Entry
9,ACWX,Momentum Entry


In [63]:
# Apply TA filters and prioritize ETFs
results = []
buy_list = buy_list[buy_list['Entry_Signal'].isin([
    'Momentum Entry',
    'Pullback Entry'
])]

#for etf in ['EWH', 'VGK'] : # buy_list['ETF'].to_list():
for etf in buy_list['ETF'].to_list():
   df =get_daily_data(etf)
   price = df['Close'].iloc[-1].iloc[0]
   above_21EMA = price > df['21_day_EMA'].iloc[-1]
   above_50sma = price  > df['50_day_SMA'].iloc[-1]
   price_21ema = df['21_day_EMA'].iloc[-1]
   price_13ema = df['13_day_EMA'].iloc[-1]



   if  above_21EMA or above_50sma :
    rr_ratio,support_level, resistance_level, latest_price,trail = calculate_risk_reward(df)
    stop_loss = price_13ema - 1.5*trail
    # Fetch the Entry_Signal from buy_list
    entry_signal = buy_list.loc[buy_list['ETF'] == etf, 'Entry_Signal'].values[0]
    # Append results with Entry_Signal
    results.append({
            "ETF": etf,
            "Risk-Reward": rr_ratio,
            "Support": support_level,
            "Resistance": resistance_level,
            "Current Price": latest_price,
            "Trail Price": trail,
            "Entry Signal": entry_signal,  # Add entry signal
            "Stop Loss": stop_loss
        })

    time.sleep(2)  # Add a delay of 1 second between requests


# Sort ETFs by highest risk-to-reward ratio
try:
   df_results = pd.DataFrame(results).dropna().sort_values(by="Risk-Reward", ascending=False).reset_index(drop = True)
except Exception as e:
  print("No ETF to buy today, check back some other time!")
  df_results = pd.DataFrame({"ETF": ["No ETF available"]})

df2 = df_results.merge(df_o[['ETF', 'score']], on='ETF', how='left')
df2 = df2.sort_values(by='score', ascending=False)
df2

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Risk-Reward,Support,Resistance,Current Price,Trail Price,Entry Signal,Stop Loss,score
15,PICB,2.0,23.088789,23.902424,23.360001,0.271212,Momentum Entry,22.635975,0.97
0,SMOG,2.0,99.796878,110.096254,103.230003,3.433125,Momentum Entry,94.173125,0.40
1,FEZ,2.0,54.530626,60.228750,56.430000,1.899375,Momentum Entry,51.420699,0.39
2,VGK,2.0,70.172499,76.735003,72.360001,2.187501,Momentum Entry,66.820127,0.38
4,VEA,2.0,50.803750,55.682501,52.430000,1.626250,Momentum Entry,48.414503,0.37
5,EWS,2.0,23.169375,25.661250,24.000000,0.830625,Momentum Entry,21.907486,0.37
6,EFA,2.0,81.490005,89.379997,84.120003,2.629997,Momentum Entry,77.456265,0.37
3,SPDW,2.0,36.403126,39.813751,37.540001,1.136875,Momentum Entry,34.689482,0.37
7,IEV,2.0,58.068749,63.322500,59.820000,1.751250,Momentum Entry,55.347852,0.35
8,VEU,2.0,60.017500,65.545001,61.860001,1.842500,Momentum Entry,57.375028,0.35


 SMC Entries

In [67]:
# Apply TA filters and prioritize ETFs
results = []
buy_list = buy_list[buy_list['Entry_Signal'].isin(['Momentum Entry','Pullback Entry'])]
tickers = buy_list['ETF'].to_list()  # You can replace this with your ETF list

import yfinance as yf
import pandas as pd

def get_fibonacci_levels(high, low):
    """Returns key Fibonacci retracement levels"""
    diff = high - low
    levels = {
        '0.0%': high,
        '23.6%': high - 0.236 * diff,
        '38.2%': high - 0.382 * diff,
        '50.0%': high - 0.5 * diff,
        '61.8%': high - 0.618 * diff,
        '78.6%': high - 0.786 * diff,
        '100.0%': low
    }
    return levels

def analyze_etf_fib(tickers):
    results = []

    for ticker in tickers:
        try:
            data = yf.download(ticker, interval='1d', period='7d')
            if data.empty:
                print(f"No data for {ticker}")
                continue

            recent_high = data['High'].max()
            recent_low = data['Low'].min()

            fib_levels = get_fibonacci_levels(recent_high, recent_low)
            current_price = data['Close'].iloc[-1].iloc[0]


            # Check if current price is between 50.0% and 78.6% retracement
            upper_bound = fib_levels['23.6%'].iloc[0]
            lower_bound = fib_levels['78.6%'].iloc[0]

            if lower_bound <= current_price <= upper_bound:
                stop_loss = current_price * 0.98  # 2% below entry
                risk = current_price - stop_loss
                take_profit = current_price + 3 * risk

                results.append({
                    'ETF': ticker,
                    'Entry': round(current_price, 2),
                    'Stop Loss': round(stop_loss, 2),
                    'Take Profit': round(take_profit, 2),
                    'In Fibonacci Buy Zone': True,
                    'Fib 50%': round(upper_bound, 2),
                    'Fib 78.6%': round(lower_bound, 2)
                })
            else:
                results.append({
                    'ETF': ticker,
                    'Entry': None,
                    'Stop Loss': None,
                    'Take Profit': None,
                    'In Fibonacci Buy Zone': False,
                    'Fib 50%': round(upper_bound, 2),
                    'Fib 78.6%': round(lower_bound, 2)
                })

        except Exception as e:
            print(f"Error analyzing {ticker}: {e}")

    return pd.DataFrame(results)

# Example usage
df = analyze_etf_fib(tickers)
#df = df[df['In Fibonacci Buy Zone']== True].reset_index(drop= True)
df


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Entry,Stop Loss,Take Profit,In Fibonacci Buy Zone,Fib 50%,Fib 78.6%
0,SMOG,NaN,NaN,NaN,False,101.45,97.30
1,FEZ,NaN,NaN,NaN,False,55.61,53.59
2,VGK,NaN,NaN,NaN,False,71.59,69.71
3,SPDW,NaN,NaN,NaN,False,37.14,36.15
4,VEA,NaN,NaN,NaN,False,51.88,50.53
5,EWS,NaN,NaN,NaN,False,23.68,22.87
6,EFA,NaN,NaN,NaN,False,83.16,80.84
7,IEV,NaN,NaN,NaN,False,59.18,57.55
8,VEU,NaN,NaN,NaN,False,61.20,59.61
9,ACWX,NaN,NaN,NaN,False,55.94,54.47
